# lab_05_fourier_isf

Goal
----
Compute the Fourier coefficients of an ISF and show:
  * reconstruction from a few harmonics,
  * the coefficient magnitude spectrum c_n,
  * how a symmetric ISF (c0 = 0) vs an asymmetric ISF (c0 != 0) differ — the c0
    term is the gateway for 1/f-noise upconversion.

Figures produced
----------------
  static/figures/isf_fourier_reconstruction.png
  static/figures/isf_fourier_coefficients.png
  static/figures/symmetric_vs_asymmetric_isf_c0.png

Corresponds to: Hajimiri-Lee (1998) Eq. (12),(24).

---

> 本 notebook 由 `scripts/make_notebooks.py` 從 `simulations/lab_05_fourier_isf.py` **自動產生**（generated snapshot，非手寫檔）。
> 權威版本是 repo 裡的 lab script；lab 更新後請重跑產生器同步。
> 執行需求：clone [isf-teaching-site](https://github.com/gmcycle7/isf-teaching-site)（要 import `simulations/common`）＋ `numpy` / `scipy` / `matplotlib`。

In [ ]:
# --- Setup：本 notebook 需要教學網站 repo 的 simulations/common 模組 ---
# 還沒有原始碼的話，先 clone repo，並把本 notebook 放在 repo 目錄樹內執行：
#     git clone https://github.com/gmcycle7/isf-teaching-site.git
# 相依套件只有三個：pip install numpy scipy matplotlib（外加 jupyter 本身）
import sys
from pathlib import Path

def _find_repo_root():
    """從目前工作目錄往上找，直到看到 simulations/common 為止。"""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "simulations" / "common").is_dir():
            return base
    raise FileNotFoundError(
        "找不到 simulations/common —— 請把本 notebook 放進 isf-teaching-site "
        "repo 目錄樹內執行（git clone https://github.com/gmcycle7/isf-teaching-site.git），"
        "或手動把 <repo>/simulations/common 加入 sys.path")

ROOT = _find_repo_root()
for _p in (str(ROOT), str(ROOT / "simulations" / "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root:", ROOT)

# CJK 字型：圖的標籤有繁體中文；找不到 CJK 字型只影響文字顯示、不影響任何數值
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm
_avail = {f.name for f in _fm.fontManager.ttflist}
_cjk = next((f for f in ["Heiti TC", "Arial Unicode MS", "STHeiti",
                         "Hiragino Sans GB", "Songti SC", "PingFang TC",
                         "Noto Sans CJK TC", "Microsoft JhengHei"]
             if f in _avail), None)
if _cjk:
    plt.rcParams["font.family"] = [_cjk, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False   # ASCII 減號，避免變方塊
print("CJK font:", _cjk or "(none found — 中文標籤可能顯示為方塊)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from isf_utils import (gamma_asymmetric, compute_fourier_coefficients,
                       reconstruct_from_fourier, gamma_rms)

# notebook 版 savefig：改成 inline 顯示。
# （原始 lab script 的 plot_utils.savefig 會把 PNG 寫進 static/figures/ 且從不
#   show()；在 notebook 裡我們直接把圖畫在 cell 輸出。）
def savefig(fig, name, verbose=True):
    plt.show()
    plt.close(fig)

In [ ]:
def make_isf(theta):
    """A richer asymmetric ISF so several harmonics are non-trivial."""
    return (-np.sin(theta) + 0.35 * np.sin(2 * theta)
            + 0.18 * np.cos(3 * theta) + 0.25)  # the +0.25 sets a non-zero c0

In [ ]:
def fig_reconstruction():
    theta = np.linspace(0, 2 * np.pi, 2000, endpoint=True)
    g = make_isf(theta)
    a0, a, b, c, ph = compute_fourier_coefficients(theta, g, n_harmonics=8)

    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    ax.plot(theta / (2 * np.pi), g, color="black", lw=2, label="original ISF")
    for N in [1, 2, 4]:
        rec = reconstruct_from_fourier(theta, a0, a[:N + 1], b[:N + 1])
        ax.plot(theta / (2 * np.pi), rec, lw=1.2, label=f"reconstruct N={N}")
    ax.axhline(0, color="gray", lw=0.6)
    ax.set_xlabel(r"phase $\theta/2\pi$")
    ax.set_ylabel(r"$\Gamma(\theta)$")
    ax.set_title("ISF Fourier reconstruction (越多 harmonic 越接近)")
    ax.legend(fontsize=8)
    savefig(fig, "isf_fourier_reconstruction.png")

In [ ]:
def fig_coefficients():
    theta = np.linspace(0, 2 * np.pi, 2000, endpoint=True)
    g = make_isf(theta)
    a0, a, b, c, ph = compute_fourier_coefficients(theta, g, n_harmonics=8)

    # check Parseval: (c0/2)^2 + sum_{n>=1} c_n^2 ?= 2 Gamma_rms^2
    # The DC harmonic enters Parseval as (c0/2)^2, not c0^2.
    parseval_lhs = c[0] ** 2 / 2 + np.sum(c[1:] ** 2)
    grms = gamma_rms(theta, g)

    fig, ax = plt.subplots(figsize=(7.6, 4.4))
    n = np.arange(len(c))
    ax.bar(n, c, color="tab:blue", width=0.6)
    ax.set_xlabel("harmonic number $n$")
    ax.set_ylabel(r"$|c_n|$")
    ax.set_title(fr"ISF coefficients   ($c_0$={c[0]:.3f}, "
                 fr"$\Gamma_{{rms}}$={grms:.3f}, "
                 fr"$(c_0/2)^2+\sum_{{n\geq1}} c_n^2$={parseval_lhs:.3f} "
                 fr"$=2\Gamma_{{rms}}^2$={2*grms**2:.3f})"
                 "\n"
                 r"DC harmonic 以 $(c_0/2)^2$ 進入 Parseval")
    for i in range(len(c)):
        ax.text(i, c[i] + 0.01, f"{c[i]:.2f}", ha="center", fontsize=7)
    savefig(fig, "isf_fourier_coefficients.png")

In [ ]:
def fig_symmetric_vs_asymmetric():
    theta = np.linspace(0, 2 * np.pi, 2000, endpoint=True)
    g_sym = np.cos(theta)               # c0 = 0 (symmetric)
    g_asym = gamma_asymmetric(theta, alpha=0.4)  # c0 = 2*alpha = 0.8

    a0s, *_ , = compute_fourier_coefficients(theta, g_sym, 4)
    a0a, *_ = compute_fourier_coefficients(theta, g_asym, 4)

    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    ax.plot(theta / (2 * np.pi), g_sym, color="tab:green",
            label=fr"symmetric $\cos\theta$ ($c_0$={abs(a0s):.2f})")
    ax.plot(theta / (2 * np.pi), g_asym, color="tab:red",
            label=fr"asymmetric $\cos\theta+0.4$ ($c_0$={abs(a0a):.2f})")
    ax.axhline(0, color="gray", lw=0.6)
    ax.fill_between(theta / (2 * np.pi), g_asym, 0.4, alpha=0.12, color="tab:red")
    ax.axhline(0.4, color="tab:red", ls=":", lw=1,
               label=r"DC of asymmetric ISF = $c_0/2$")
    ax.set_xlabel(r"phase $\theta/2\pi$")
    ax.set_ylabel(r"$\Gamma(\theta)$")
    ax.set_title(r"$c_0\neq 0$ 才會把 1/f noise upconvert 到 close-in phase noise")
    ax.legend(fontsize=8)
    savefig(fig, "symmetric_vs_asymmetric_isf_c0.png")

In [ ]:
def main():
    print("[lab_05] ISF Fourier coefficients ...")
    fig_reconstruction()
    fig_coefficients()
    fig_symmetric_vs_asymmetric()

In [ ]:
# 執行整個 lab（對應原 script 的 __main__）
main()